In [ ]:
"""
Script para clasificar escarabajos de una carpeta mezclada (sin organizar por especie).
"""

import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import pandas as pd

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

MODEL_PATH = 'model_weights/beetle_classifier_resnet50_train_aug_val_real_test_4.pth'
INPUT_FOLDER = 'carpeta_con_escarabajos_mezclados/'  # Carpeta con imágenes mezcladas
OUTPUT_CSV = 'clasificaciones.csv'  # Archivo CSV con resultados
CONFIDENCE_THRESHOLD = 0.6  # Umbral para marcar como "desconocido"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# FUNCIONES
# ============================================================================

def load_model(model_path, device):
    """Carga el modelo."""
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    if isinstance(checkpoint, dict):
        state_dict = checkpoint.get('model_state_dict', checkpoint)
        class_names = checkpoint.get('class_names', None)
    else:
        state_dict = checkpoint
        class_names = None
    
    # Detectar número de clases
    if 'fc.3.weight' in state_dict:
        num_classes = state_dict['fc.3.weight'].shape[0]
    else:
        num_classes = state_dict['fc.weight'].shape[0]
    
    # Crear modelo
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(
        nn.Linear(2048, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    return model, class_names


def predict_image(model, image_path, class_names, device, transform, threshold=0.6):
    """Predice la clase de una imagen con umbral de confianza."""
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]
        top_probs, top_indices = torch.topk(probabilities, k=min(3, len(class_names)))
    
    top_probs = top_probs.cpu().numpy()
    top_indices = top_indices.cpu().numpy()
    
    predicted_class = class_names[top_indices[0]]
    confidence = top_probs[0]
    
    # Si confianza baja → marcar como unknown
    if confidence < threshold and predicted_class != 'unknown_beetle':
        status = 'low_confidence'
    else:
        status = 'ok'
    
    top_3 = [(class_names[idx], float(prob)) for idx, prob in zip(top_indices, top_probs)]
    
    return predicted_class, float(confidence), top_3, status


def classify_folder(model, folder_path, class_names, device, transform, threshold=0.6):
    """Clasifica todas las imágenes de una carpeta."""
    results = []
    
    print(f"\n🔄 Clasificando imágenes de: {folder_path}")
    
    image_files = [f for f in os.listdir(folder_path) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f"📊 {len(image_files)} imágenes encontradas\n")
    
    for img_file in image_files:
        img_path = os.path.join(folder_path, img_file)
        
        try:
            pred_class, confidence, top_3, status = predict_image(
                model, img_path, class_names, device, transform, threshold
            )
            
            # Mostrar resultado
            if status == 'low_confidence':
                print(f"⚠️  {img_file:40s} → {pred_class:25s} ({confidence*100:5.1f}%) [BAJA CONFIANZA]")
            else:
                print(f"✅ {img_file:40s} → {pred_class:25s} ({confidence*100:5.1f}%)")
            
            results.append({
                'image': img_file,
                'predicted_class': pred_class,
                'confidence': confidence,
                'status': status,
                'top_1_class': top_3[0][0],
                'top_1_prob': top_3[0][1],
                'top_2_class': top_3[1][0] if len(top_3) > 1 else '',
                'top_2_prob': top_3[1][1] if len(top_3) > 1 else 0,
                'top_3_class': top_3[2][0] if len(top_3) > 2 else '',
                'top_3_prob': top_3[2][1] if len(top_3) > 2 else 0,
            })
            
        except Exception as e:
            print(f"❌ Error procesando {img_file}: {e}")
            results.append({
                'image': img_file,
                'predicted_class': 'ERROR',
                'confidence': 0,
                'status': 'error'
            })
    
    return results


def main():
    """Función principal."""
    print("="*80)
    print("🪲 CLASIFICADOR DE ESCARABAJOS - CARPETA MEZCLADA")
    print("="*80)
    
    # Transformaciones
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Cargar modelo
    print(f"\n📥 Cargando modelo...")
    model, class_names = load_model(MODEL_PATH, DEVICE)
    print(f"✅ Modelo cargado: {len(class_names)} clases")
    
    # Clasificar carpeta
    results = classify_folder(
        model, INPUT_FOLDER, class_names, DEVICE, transform, CONFIDENCE_THRESHOLD
    )
    
    # Guardar resultados en CSV
    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\n{'='*80}")
    print(f"📊 RESUMEN")
    print(f"{'='*80}")
    print(f"Total de imágenes: {len(results)}")
    print(f"Clasificaciones OK: {len([r for r in results if r['status'] == 'ok'])}")
    print(f"Baja confianza: {len([r for r in results if r['status'] == 'low_confidence'])}")
    print(f"Errores: {len([r for r in results if r['status'] == 'error'])}")
    print(f"\n✅ Resultados guardados en: {OUTPUT_CSV}")
    print(f"{'='*80}")
    
    # Mostrar distribución por especie
    print(f"\n📊 Distribución por especie:")
    print(f"{'-'*80}")
    species_count = df[df['status'] != 'error']['predicted_class'].value_counts()
    for species, count in species_count.items():
        print(f"  {species:35s}: {count:3d} imágenes")
    print(f"{'-'*80}")


if __name__ == "__main__":
    main()